# Build Dynamic Datasets

Dans ce notebook, la cohorte de développement est utilisée pour préparer les jeux de données destinés à l'entraînement des modèles. Elle est séparée en ensembles d'entraînement et de test, puis des snapshots longitudinaux sont construits à partir de l'historique disponible à chaque visite.

À l'issue de cette étape, les jeux de données dynamiques `train_dynamic.parquet` et `internal_test_dynamic.parquet` sont générés. Ils serviront de point de départ à la préparation spécifique de chaque modèle de survie.

In [ ]:
# ==========================================================
# LIBRAIRIES
# ==========================================================

# Data manipulation
import numpy as np
import pandas as pd
import polars as pl

from pathlib import Path

# Machine learning
from sklearn.model_selection import train_test_split

In [ ]:
# ==========================================================
# IMPORTATION DES DONNÉES
# ==========================================================

df_development = pl.read_parquet(
    "My_data/adpkd_longitudinal_development.parquet"
)

# La cohorte de validation externe sera importée
# uniquement au moment de la validation finale.

df_external_test = pl.read_parquet(
    "My_data/adpkd_longitudinal_external_test.parquet"
)

### Split train test

In [ ]:
# ==========================================================
# REPRODUCTIBILITÉ
# ==========================================================

SEED = 42

# ==========================================================
# SÉPARATION TRAIN / TEST AU NIVEAU DES PATIENTS
# ==========================================================

# La cohorte de développement est séparée en un jeu
# d'entraînement (80 %) et un jeu de test (20 %).
#
# Le découpage est réalisé au niveau des patients afin
# d'éviter toute fuite d'information entre les deux jeux.
# La stratification est effectuée sur l'événement IRT afin
# de préserver sa proportion dans chaque cohorte.

patients = (
    df_development
    .select(["patient_id", "event_irt"])
    .group_by("patient_id")
    .agg(
        pl.max("event_irt").alias("event_irt")
    )
    .sort("patient_id")
    .to_pandas()
)

# Split stratifié 80 / 20
train_ids, test_ids = train_test_split(
    patients,
    test_size=0.2,
    stratify=patients["event_irt"],
    random_state=SEED
)

# Cohorte d'entraînement

df_train = df_development.filter(
    pl.col("patient_id").is_in(train_ids["patient_id"])
)

# Cohorte de test

df_test = df_development.filter(
    pl.col("patient_id").is_in(test_ids["patient_id"])
)

In [ ]:
# ==========================================================
# RÉSUMÉ D'UNE COHORTE
# ==========================================================

def summarize_split(df, name):
    """
    Affiche les caractéristiques principales d'une cohorte.
    """

    patient_summary = (
        df
        .group_by("patient_id")
        .agg(
            pl.max("event_irt").alias("event_irt")
        )
    )

    n_patients = patient_summary.height
    n_events = (
        patient_summary
        .filter(pl.col("event_irt") == 1)
        .height
    )

    event_rate = n_events / n_patients
    n_observations = df.height

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print(f"Patients       : {n_patients}")
    print(f"Observations   : {n_observations}")
    print(f"Événements IRT : {n_events}")
    print(f"Taux IRT       : {event_rate:.2%}")

In [ ]:
# ==========================================================
# RÉSUMÉ DES COHORTES
# ==========================================================

summarize_split(df_development, "DEVELOPMENT")

summarize_split(df_train, "TRAIN")

summarize_split(df_test, "TEST")

summarize_split(df_external_test, "TEST EXTERNE")

#### CONSTRUCTION DES SNAPSHOTS LONGITUDINAUX

In [ ]:
def build_dynamic_dataset(df):
    """
    Construit un dataset longitudinal dynamique.

    Chaque ligne correspond à un snapshot créé à partir
    d'une visite. Les variables sont calculées uniquement
    à partir des informations disponibles jusqu'à cette
    visite, sans utiliser les observations futures.
    """

    snapshots = []

    for patient_id, patient_df in df.group_by("patient_id"):

        # Tri chronologique des visites
        patient_df = patient_df.sort("creat_date")

        n_obs = patient_df.height

        # Construction d'un snapshot à chaque visite
        for i in range(n_obs):

            current_visit = patient_df.row(i, named=True)

            # --------------------------------------------------
            # Nombre cumulé de mesures disponibles
            # au moment de la visite courante
            # --------------------------------------------------

            n_measures = i + 1

            # --------------------------------------------------
            # Historique DFG disponible
            # --------------------------------------------------

            dfg_history = (
                patient_df["ckd_epi"][:n_measures]
                .to_list()
            )

            # Tableau NumPy utilisé pour les calculs numériques
            egfr = np.asarray(
                dfg_history,
                dtype=float
            )

            # --------------------------------------------------
            # Historique temporel (années depuis la première mesure)
            # --------------------------------------------------

            time_history = (
                (
                    patient_df["creat_date"][:n_measures]
                    - patient_df["creat_date"][0]
                )
                .dt.total_days()
                .to_numpy()
                / 365.25
            )

            # Liste conservée pour les modèles séquentiels
            time_history_years = (
                time_history
                .tolist()
            )

            # --------------------------------------------------
            # Historique des âges (temps absolu)
            # --------------------------------------------------

            age_history = (
                patient_df["age_creat"][:n_measures]
                .to_list()
            )

            # --------------------------------------------------
            # Historique HTA
            # --------------------------------------------------

            raw_hypertension_history  = (
                patient_df["hypertension_at_t"][:n_measures]
                .to_list()
            )

            hypertension_age_history = (
                patient_df["age_hypertension_at_t"][:n_measures]
                .to_list()
            )

            # Présence d'une hypertension (0 = non, 1 = oui)

            hypertension_history   = [
                int(x > 0)
                for x in raw_hypertension_history
            ]

            # Date de survenue connue
            # 1 = date connue
            # 0 = absence d'HTA ou date inconnue

            hypertension_date_known_history  = [
                int(x == 1)
                for x in raw_hypertension_history
            ]

            # Nombre d'années depuis l'apparition de l'HTA

            years_since_hypertension_history = []

            for age_visit, age_ht in zip(age_history, hypertension_age_history):

                if age_ht is None:
                   years_since_hypertension_history.append(0.0)

                else:
                    years_since_hypertension_history.append(
                        float(age_visit - age_ht)
                    )

            # --------------------------------------------------
            # Historique des complications urologiques
            # --------------------------------------------------

            raw_urologic_history = (
                patient_df["urologic_event_at_t"][:n_measures]
                .to_list()
            )

            urologic_age_history = (
                patient_df["age_urologic_event_at_t"][:n_measures]
                .to_list()
            )
            # Présence d'une hypertension (0 = non, 1 = oui)

            urologic_history = [
                int(x > 0)
                for x in raw_urologic_history
            ]

            # Date de survenue connue
            # 1 = date connue
            # 0 = absence d'HTA ou date inconnue

            urologic_date_known_history   = [
                int(x == 1)
                for x in raw_urologic_history
            ]

            # Nombre d'années depuis la première complication

            years_since_urologic_event_history = []

            for age_visit, age_uro in zip(age_history, urologic_age_history):

                if age_uro is None:
                   years_since_urologic_event_history.append(0.0)

                else:
                     years_since_urologic_event_history.append(
                         float(age_visit - age_uro)
                     )

            
            # --------------------------------------------------
            # Temps écoulé depuis la première mesure
            # --------------------------------------------------

            time_since_first_measure = (
                time_history[-1]
            )

            # --------------------------------------------------
            # Conditions de calcul des pentes
            # --------------------------------------------------

            can_compute_global_slope = (
                (n_measures >= 3)
                and (time_since_first_measure >= 0.25)
            )

            can_compute_recent_slope = (
                (n_measures >= 3)
                and (
                    time_history[-1]
                    - time_history[-3]
                    >= 0.5
                )
            )

            # --------------------------------------------------
            # Pente cumulée du DFG
            # --------------------------------------------------

            if can_compute_global_slope:

                egfr_slope = float(
                    np.polyfit(
                        time_history,
                        egfr,
                        deg=1
                    )[0]
                )

            else:

                egfr_slope = np.nan

            # --------------------------------------------------
            # Pente récente (3 dernières mesures)
            # --------------------------------------------------

            if can_compute_recent_slope:

                recent_egfr_slope = float(
                    np.polyfit(
                        time_history[-3:],
                        egfr[-3:],
                        deg=1
                    )[0]
                )

            else:

                recent_egfr_slope = np.nan

            # --------------------------------------------------
            # DFG de référence
            # --------------------------------------------------

            baseline_ckd_epi = (
                egfr[0]
            )

            # --------------------------------------------------
            # DFG précédent
            # --------------------------------------------------

            previous_ckd_epi = (
                current_visit["ckd_epi"]
                if n_measures == 1
                else egfr[-2]
            )
            
            # --------------------------------------------------
            # Statistiques du DFG
            # --------------------------------------------------

            egfr_mean = float(
                egfr.mean()
            )

            egfr_std = (
                float(
                    egfr.std(ddof=1)
                )
                if n_measures >= 2
                else np.nan
            )

            egfr_min = float(
                egfr.min()
            )

            egfr_max = float(
                egfr.max()
            )

            egfr_cv = (
                egfr_std / egfr_mean
                if (
                    n_measures >= 2
                    and egfr_mean > 0
                )
                else np.nan
            )

            egfr_iqr = (
                float(
                    np.percentile(egfr, 75)
                    - np.percentile(egfr, 25)
                )
                if n_measures >= 4
                else np.nan
            )

            # --------------------------------------------------
            # Variation depuis la baseline
            # --------------------------------------------------

            delta_ckd_from_baseline = (
                current_visit["ckd_epi"]
                - baseline_ckd_epi
            )

            percent_decline_from_baseline = (
                100
                * delta_ckd_from_baseline
                / baseline_ckd_epi
                if baseline_ckd_epi > 0
                else np.nan
            )

            # --------------------------------------------------
            # Variation récente
            # --------------------------------------------------

            recent_delta_ckd = (
                egfr[-1] - egfr[-2]
                if n_measures >= 2
                else np.nan
            )

            # --------------------------------------------------
            # Intervalles entre mesures
            # --------------------------------------------------

            measurement_intervals = np.diff(time_history)

            # --------------------------------------------------
            # Statistiques des intervalles
            # --------------------------------------------------

            if measurement_intervals.size == 0:

                time_since_last_measure = np.nan
                mean_measurement_interval = np.nan
                std_measurement_interval = np.nan

            else:

                time_since_last_measure = float(
                    measurement_intervals[-1]
                )

                mean_measurement_interval = float(
                    measurement_intervals.mean()
                )

                std_measurement_interval = (
                    float(
                        measurement_intervals.std(ddof=1)
                    )
                    if measurement_intervals.size >= 2
                    else np.nan
                )


            # --------------------------------------------------
            # Variation annuelle récente
            # --------------------------------------------------

            if (
                measurement_intervals.size > 0
                and measurement_intervals[-1] > 0
            ):

                recent_annualized_change = (
                    recent_delta_ckd
                    / measurement_intervals[-1]
                )

            else:

                recent_annualized_change = np.nan

            #------------------------------------------------
            # Liste conservée pour les modèles séquentiels
            # -----------------------------------------------
            
            measurement_intervals = (
                measurement_intervals.tolist()
            )

            # --------------------------------------------------
            # Densité du suivi
            # --------------------------------------------------

            followup_density = (
                (n_measures - 1)
                / time_since_first_measure
                if time_since_first_measure >= 0.25
                else np.nan
            )

            # --------------------------------------------------
            # Résumé clinique
            # --------------------------------------------------

            ever_hypertension = int(
                current_visit["hypertension_at_t"] > 0
            )

            ever_urologic_event = int(
                current_visit["urologic_event_at_t"] > 0
            )

            # --------------------------------------------------
            # Temps depuis l'hypertension
            # --------------------------------------------------

            if (
                current_visit["age_hypertension_at_t"] is not None
            ):

                years_since_hypertension = (
                    current_visit["age_creat"]
                    - current_visit["age_hypertension_at_t"]
                )

            else:

                years_since_hypertension = np.nan

            # --------------------------------------------------
            # Temps depuis la première complication urologique
            # --------------------------------------------------

            if (
                current_visit["age_urologic_event_at_t"] is not None
            ):

                years_since_urologic_event = (
                    current_visit["age_creat"]
                    - current_visit["age_urologic_event_at_t"]
                )

            else:

                years_since_urologic_event = np.nan

            # --------------------------------------------------
            # Position dans le suivi
            # --------------------------------------------------

            is_first_measure = int(
                n_measures == 1
            )

            is_second_measure = int(
                n_measures == 2
            )

            has_longitudinal_history = int(
                n_measures >= 3
            )

            # --------------------------------------------------
            # Âge à la première mesure
            # --------------------------------------------------

            baseline_age = (
                patient_df["age_creat"][0]
            )
            
            # --------------------------------------------------
            # Snapshot
            # --------------------------------------------------

            snapshots.append({

                # ==========================================
                # IDENTIFIANT
                # ==========================================

                "patient_id":
                    current_visit["patient_id"],

                # ==========================================
                # VARIABLES FIXES
                # ==========================================

                "sex":
                    current_visit["sex"],

                "genetic":
                    current_visit["genetic"],

                # ==========================================
                # VARIABLES CLINIQUES ACTUELLES
                # ==========================================

                "age_creat":
                    current_visit["age_creat"],

                "hypertension_at_t":
                    current_visit["hypertension_at_t"],

                "age_hypertension_at_t":
                    current_visit["age_hypertension_at_t"],

                "urologic_event_at_t":
                    current_visit["urologic_event_at_t"],

                "age_urologic_event_at_t":
                    current_visit["age_urologic_event_at_t"],

                # ==========================================
                # VARIABLES RÉNALES ACTUELLES
                # ==========================================

                "current_ckd_epi":
                    current_visit["ckd_epi"],

                # ==========================================
                # HISTORIQUES (ODE-RNN / CDE)
                # ==========================================

                "egfr_history":
                    dfg_history,

                "time_history":
                    time_history_years,

                "age_history":
                    age_history,


                "hypertension_age_history":
                    hypertension_age_history,


                "urologic_age_history":
                    urologic_age_history,

                "measurement_intervals":
                    measurement_intervals,

                # ==========================================
                # VARIABLES DYNAMIQUES
                # ==========================================

                "n_measures":
                    n_measures,

                "time_since_first_measure":
                    time_since_first_measure,

                "time_since_last_measure":
                    time_since_last_measure,

                "followup_density":
                    followup_density,

                # ==========================================
                # FEATURES LONGITUDINALES
                # ==========================================

                "baseline_ckd_epi":
                    baseline_ckd_epi,

                "previous_ckd_epi":
                    previous_ckd_epi,

                "egfr_slope":
                    egfr_slope,

                "recent_egfr_slope":
                    recent_egfr_slope,

                "recent_annualized_change":
                    recent_annualized_change,

                "egfr_mean":
                    egfr_mean,

                "egfr_std":
                    egfr_std,

                "egfr_min":
                    egfr_min,

                "egfr_max":
                    egfr_max,

                "egfr_cv":
                    egfr_cv,

                "egfr_iqr":
                    egfr_iqr,

                "delta_ckd_from_baseline":
                    delta_ckd_from_baseline,

                "percent_decline_from_baseline":
                    percent_decline_from_baseline,

                "recent_delta_ckd":
                    recent_delta_ckd,

                "mean_measurement_interval":
                    mean_measurement_interval,

                "std_measurement_interval":
                    std_measurement_interval,

                "baseline_age":
                    baseline_age,

                "ever_hypertension":
                    ever_hypertension,

                "ever_urologic_event":
                    ever_urologic_event,

                "years_since_hypertension":
                    years_since_hypertension,

                "years_since_urologic_event":
                    years_since_urologic_event,

                "hypertension_history":
                    hypertension_history,

                "hypertension_date_known_history":
                    hypertension_date_known_history,

                "years_since_hypertension_history":
                    years_since_hypertension_history,

                "urologic_history":
                    urologic_history,

                "urologic_date_known_history":
                    urologic_date_known_history,

                "years_since_urologic_event_history":
                    years_since_urologic_event_history,

                "is_first_measure":
                    is_first_measure,

                "is_second_measure":
                    is_second_measure,

                "has_longitudinal_history":
                    has_longitudinal_history,

                # ==========================================
                # INFORMATIONS DU SNAPSHOT
                # ==========================================

                "creat_date":
                    current_visit["creat_date"],

                # ==========================================
                # CIBLE
                # ==========================================

                "time_to_event":
                    current_visit["time_to_event"],

                "event_irt":
                    current_visit["event_irt"]

            })

    return pl.DataFrame(snapshots)

In [ ]:
# ==========================================================
# CONSTRUCTION DES JEUX DYNAMIQUES
# ==========================================================

# Jeu d'entraînement
train_dynamic = build_dynamic_dataset(
    df_train
)

# Jeu de test interne
internal_test_dynamic = build_dynamic_dataset(
    df_test
)

# Jeu de validation externe
external_test_dynamic = build_dynamic_dataset(
    df_external_test
)

In [ ]:
# ==========================================================
# RÉSUMÉ DES JEUX DYNAMIQUES
# ==========================================================

print("\n=========== Jeux dynamiques ===========\n")

print(
    f"Train               : {train_dynamic.height} snapshots | "
    f"{train_dynamic.select(pl.col('patient_id').n_unique()).item()} patients"
)

print(
    f"Test interne        : {internal_test_dynamic.height} snapshots | "
    f"{internal_test_dynamic.select(pl.col('patient_id').n_unique()).item()} patients"
)

print(
    f"Validation externe  : {external_test_dynamic.height} snapshots | "
    f"{external_test_dynamic.select(pl.col('patient_id').n_unique()).item()} patients"
)

In [ ]:
# ==========================================================
# SAUVEGARDE DES JEUX DYNAMIQUES
# ==========================================================

train_dynamic.write_parquet(
    "My_data/train_dynamic.parquet"
)

internal_test_dynamic.write_parquet(
    "My_data/internal_test_dynamic.parquet"
)

external_test_dynamic.write_parquet(
    "My_data/external_test_dynamic.parquet"
)

print("Jeux dynamiques sauvegardés avec succès.")